In [0]:
# 04_gold/4_gold_transform.py
# GOLD: Modelo Estrella

print("="*50)
print("GOLD: Modelo Estrella")
print("="*50)

df_silver = spark.table("capa_silver.coffee_sales_silver")

from pyspark.sql.functions import col, lit

# =====================================================
# DIMENSIONES
# =====================================================

print(" Creando dimensiones...")

# 1. PRODUCTO
dim_product = df_silver.select("Product_ID", "coffee_name") \
    .distinct() \
    .withColumnRenamed("Product_ID", "Product_Key") \
    .withColumnRenamed("coffee_name", "Product_Name")
dim_product.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("capa_gold.dim_product")
print("   dim_product")

# 2. FECHA
dim_date = df_silver.select("Date_ID", "Date", "Year", "Month_Number", "Month_Name_Full", "Day", "Day_Of_Week") \
    .distinct() \
    .withColumnRenamed("Date_ID", "Date_Key") \
    .withColumnRenamed("Date", "Date") \
    .withColumnRenamed("Year", "Year") \
    .withColumnRenamed("Month_Number", "Month") \
    .withColumnRenamed("Month_Name_Full", "Month_Name") \
    .withColumnRenamed("Day", "Day") \
    .withColumnRenamed("Day_Of_Week", "Day_Of_Week")
dim_date.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("capa_gold.dim_date")
print("   dim_date")

# 3. HORA (SOLO 3 VALORES)
dim_time = df_silver.select("Time_ID", "Time_Period") \
    .distinct() \
    .withColumnRenamed("Time_ID", "Time_Key") \
    .withColumnRenamed("Time_Period", "Time_Period")
dim_time.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("capa_gold.dim_time")
print("   dim_time (solo 3 valores)")

# 4. DÍA
dim_day = df_silver.select("Day_ID", "Weekday", "is_weekend") \
    .distinct() \
    .withColumnRenamed("Day_ID", "Day_Key") \
    .withColumnRenamed("Weekday", "Day_Name") \
    .withColumnRenamed("is_weekend", "Is_Weekend")
dim_day.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("capa_gold.dim_day")
print("   dim_day")

# 5. MES
dim_month = df_silver.select("Month_ID", "Month_name", "Monthsort") \
    .distinct() \
    .withColumnRenamed("Month_ID", "Month_Key") \
    .withColumnRenamed("Month_name", "Month_Name") \
    .withColumnRenamed("Monthsort", "Month_Number")
dim_month.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("capa_gold.dim_month")
print("   dim_month")

# 6. TEMPORADA
dim_season = df_silver.select("Season_ID", "Season") \
    .distinct() \
    .withColumnRenamed("Season_ID", "Season_Key") \
    .withColumnRenamed("Season", "Season_Name")
dim_season.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("capa_gold.dim_season")
print("   dim_season")

# 7. PAGO
dim_payment = df_silver.select("Payment_ID", "cash_type") \
    .distinct() \
    .withColumnRenamed("Payment_ID", "Payment_Key") \
    .withColumnRenamed("cash_type", "Payment_Method")
dim_payment.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("capa_gold.dim_payment")
print("   dim_payment")

# =====================================================
# TABLA DE HECHOS
# =====================================================

print(" Creando tabla de hechos...")

fact_sales = df_silver.select(
    col("Product_ID").alias("Product_Key"),
    col("Date_ID").alias("Date_Key"),
    col("Time_ID").alias("Time_Key"),
    col("Day_ID").alias("Day_Key"),
    col("Month_ID").alias("Month_Key"),
    col("Season_ID").alias("Season_Key"),
    col("Payment_ID").alias("Payment_Key"),
    col("money").alias("Total_Revenue"),
    lit(1).alias("Quantity_Sold")
)

fact_sales.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("capa_gold.fact_sales")
print("   fact_sales")

print("="*50)
print(" MODELO ESTRELLA COMPLETADO")
print("="*50)

print("\n VERIFICACIÓN DE DIM_TIME (SOLO 3 VALORES):")
display(spark.table("capa_gold.dim_time"))

print("\n VERIFICACIÓN DE DIM_PAYMENT (CARD Y CASH):")
display(spark.table("capa_gold.dim_payment"))

print("\n Ejemplo fact_sales:")
display(spark.table("capa_gold.fact_sales").limit(5))

GOLD: Modelo Estrella
 Creando dimensiones...
   dim_product
   dim_date
   dim_time (solo 3 valores)
   dim_day
   dim_month
   dim_season
   dim_payment
 Creando tabla de hechos...
   fact_sales
 MODELO ESTRELLA COMPLETADO

 VERIFICACIÓN DE DIM_TIME (SOLO 3 VALORES):


Time_Key,Time_Period
1,Morning
3,Night
2,Afternoon



 VERIFICACIÓN DE DIM_PAYMENT (CARD Y CASH):


Payment_Key,Payment_Method
2,cash
1,card



 Ejemplo fact_sales:


Product_Key,Date_Key,Time_Key,Day_Key,Month_Key,Season_Key,Payment_Key,Total_Revenue,Quantity_Sold
2,1,3,1,3,1,1,33.8,1
1,2,1,3,3,1,1,28.9,1
7,3,2,4,3,1,1,38.7,1
2,7,2,5,3,1,1,33.8,1
3,8,2,1,3,1,1,38.7,1
